In [ ]:
# Supplementary Figure 15: NT-specific synapse distance relative to a neuropil-matched null
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import betabinom

contact_dir = Path('data/contact_distances')
out_dir = Path('supp_outputs')
out_dir.mkdir(parents=True, exist_ok=True)
manifest = json.loads((contact_dir / 'manifest.json').read_text())
nt_order = ['ACH', 'GABA', 'GLUT', 'DA', 'SER', 'OCT']
n_perm = 5000
rng = np.random.default_rng(20260821)

def safe_name(name):
    return re.sub(r'[^A-Za-z0-9._-]+', '_', str(name)).strip('_')

def read_distances(path):
    return np.fromfile(path, dtype=np.float32)

def null_medians(population, n):
    population = np.sort(population)
    total = len(population)
    if n == total:
        return np.full(n_perm, np.median(population), dtype=float)
    if n % 2:
        k = (n + 1) // 2
        rank = k + betabinom.rvs(total-n, k, n-k+1, size=n_perm, random_state=rng)
        return population[rank-1]
    k = n // 2
    rank1 = k + betabinom.rvs(total-n, k, n-k+1, size=n_perm, random_state=rng)
    gap = betabinom.rvs(total-rank1-(n-k), 1, n-k, size=n_perm, random_state=rng)
    rank2 = rank1 + 1 + gap
    return (population[rank1-1].astype(float) + population[rank2-1].astype(float))/2

def bh_fdr(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adjusted = np.minimum.accumulate((p[order] * len(p) / np.arange(1,len(p)+1))[::-1])[::-1]
    q = np.empty_like(p)
    q[order] = np.minimum(adjusted, 1)
    return q

rows = []
nt_groups = set(map(tuple, manifest['nt_groups']))
for neuropil in manifest['regions']:
    all_contacts = read_distances(contact_dir / 'all' / f'{safe_name(neuropil)}.f32')
    for nt in nt_order:
        if (neuropil, nt) not in nt_groups:
            continue
        contacts = read_distances(contact_dir / 'nt' / f'{safe_name(neuropil)}__{nt}.f32')
        observed = float(np.median(contacts))
        draws = null_medians(all_contacts, len(contacts))
        expected = float(np.median(draws))
        tail_left = (1 + np.count_nonzero(draws <= observed)) / (n_perm+1)
        tail_right = (1 + np.count_nonzero(draws >= observed)) / (n_perm+1)
        rows.append({'neuropil': neuropil, 'nt_type': nt, 'n': len(contacts),
                     'observed_median_um': observed, 'null_median_um': expected,
                     'observed_minus_null_um': observed-expected,
                     'p_two_sided': min(1, 2*min(tail_left,tail_right))})
results = pd.DataFrame(rows)
results['q_fdr'] = bh_fdr(results['p_two_sided'].to_numpy())

pivot = results.pivot(index='neuropil', columns='nt_type', values='observed_minus_null_um')
q_pivot = results.pivot(index='neuropil', columns='nt_type', values='q_fdr')
order = [n for n in manifest['regions'] if n in pivot.index]
pivot = pivot.reindex(index=order, columns=nt_order)
q_pivot = q_pivot.reindex(index=order, columns=nt_order)
effects = pivot.to_numpy(float)
limit = max(float(np.nanmax(np.abs(effects))), .1)
fig, ax = plt.subplots(figsize=(7, max(5, .25*len(order)+1)))
image = ax.imshow(effects, aspect='auto', cmap='RdBu_r', norm=TwoSlopeNorm(vcenter=0, vmin=-limit, vmax=limit))
for (i, j), q in np.ndenumerate(q_pivot.to_numpy(float)):
    if q < .05:
        ax.text(j, i, '*', ha='center', va='center', fontsize=10)
ax.set_xticks(np.arange(len(nt_order)), nt_order)
ax.set_yticks(np.arange(len(order)), order, fontsize=8)
ax.tick_params(length=0)
fig.colorbar(image, ax=ax, label='Observed − null median distance (µm)')
plt.rcParams['svg.fonttype'] = 'none'
fig.tight_layout()
fig.savefig(out_dir / 'supp_fig15.svg', bbox_inches='tight')
plt.show()
print(results.groupby('nt_type')['q_fdr'].apply(lambda x: int((x<.05).sum())))